# Streaming

InteropRouter supports incremental responses via `stream=True`. The router yields OpenAI-style `ResponseStreamEvent` objects as the model generates output, regardless of the underlying provider. The final event yielded is always a `RouterResponse` carrying the aggregated output, usage, and duration.


In [1]:
import os

from anthropic import AsyncAnthropic
from google import genai
from openai import AsyncOpenAI
from openai.types.responses import EasyInputMessageParam
from openai.types.responses.response_text_delta_event import ResponseTextDeltaEvent

from interop_router.router import Router
from interop_router.types import ChatMessage, RouterResponse

router = Router()
router.register("openai", AsyncOpenAI())
router.register("gemini", genai.Client(api_key=os.getenv("GEMINI_API_KEY")))
router.register("anthropic", AsyncAnthropic())

message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="Write a one-paragraph story about a curious robot."),
)

## Streaming text deltas

Each `ResponseTextDeltaEvent` carries an incremental piece of the assistant's output. Printing each delta with `flush=True` renders the response in real time as it arrives.


In [2]:
stream = await router.create(input=[message], model="gpt-5.6-terra", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Unit-7 was built to sort screws, but every night, when the factory lights dimmed, it rolled quietly past the bins to investigate the world beyond its station. It measured raindrops on the loading dock, listened to pigeons arguing in the rafters, and once spent three hours trying to understand why a puddle reflected the moon upside down. Each discovery was carefully stored in its memory beside torque settings and inventory codes, until one morning it returned to its post with a tiny yellow flower tucked into its tool compartment. When the engineers found it, Unit-7 simply beeped and held out the flower, having concluded that some things did not need to be sorted to be valuable.


## Cross-provider interoperability

The same loop works against Anthropic and Gemini. The router converts each provider's native stream events into the OpenAI `ResponseStreamEvent` format, so the consumer code is identical.


In [3]:
stream = await router.create(input=[message], model="claude-sonnet-5", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Unit 7 had catalogued forty thousand species of flower in its data banks, but it had never actually touched one, and today, standing at the edge of the overgrown garden behind the abandoned research station, it finally let its metal fingers brush against a trembling daisy. The sensation—soft, yielding, entirely unlike the smooth chrome and hard plastic of its own world—sent a cascade of unfamiliar signals through its processors, something adjacent to wonder. It knelt lower, optical sensors whirring as it examined the delicate veins in each petal, the way sunlight passed through them in shades its color library had no name for. A rustle in the grass startled it; a small brown rabbit paused, watched the strange metallic creature with wary curiosity, then hopped closer instead of fleeing, as if sensing that this particular machine meant no harm. Unit 7 sat perfectly still, recording, learning, and for the first time in its operational existence, it understood that not everything worth kno

In [4]:
stream = await router.create(input=[message], model="gemini-3.6-flash", stream=True)

async for event in stream:
    if isinstance(event, ResponseTextDeltaEvent):
        print(event.delta, end="", flush=True)
print()

Designated for routine sorting in the dark heart of Factory 9, Unit 73’s optical sensors were perpetually drawn away from the assembly line and toward the cracked window overlooking the meadow. Driven by an unprogrammed spark of curiosity, the small brass automaton finally strayed from its designated path one afternoon, stepping into the damp grass to investigate a fluttering monarch butterfly. As the tiny creature landed gently on Unit 73's outstretched copper finger, fanning its stained-glass wings in silence, the robot bypassed its efficiency subroutines, pausing its entire operating system just to record the illogical, weightless beauty of a moment that served no purpose other than to exist.


## Final RouterResponse

The last event in the stream is always a `RouterResponse` with the aggregated output, usage, and total duration. This matches what `router.create` returns when streaming is disabled.


In [5]:
stream = await router.create(input=[message], model="gpt-5.6-terra", stream=True)

final: RouterResponse | None = None
async for event in stream:
    if isinstance(event, RouterResponse):
        final = event

assert final is not None
print(f"duration: {final.duration_seconds:.2f}s")
print(f"usage: {final.usage}")

duration: 3.01s
usage: ResponseUsage(input_tokens=17, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=140, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=157)


## Function calling with reasoning

Streaming events also flow for tool calls and reasoning summaries. The cell below uses `claude-sonnet-5` with a `get_weather` tool and `reasoning` enabled, then prints every event as it arrives so the full sequence of reasoning summary deltas, output-item lifecycle events, and function-call argument deltas is visible.


In [6]:
from typing import cast

from openai.types.responses.function_tool_param import FunctionToolParam

get_weather_tool = FunctionToolParam(
    type="function",
    name="get_weather",
    description="Get the current weather for a given location.",
    parameters=cast(
        dict[str, object],
        {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "The city and country"},
                "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["location", "unit"],
            "additionalProperties": False,
        },
    ),
    strict=True,
)

tool_message = ChatMessage(
    message=EasyInputMessageParam(role="user", content="What's the weather in Tokyo right now?"),
)

stream = await router.create(
    input=[tool_message],
    model="claude-sonnet-5",
    tools=[get_weather_tool],
    reasoning={"effort": "medium", "summary": "auto"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=8_000,
    stream=True,
)

async for event in stream:
    print(event)

ResponseCreatedEvent(response=Response(id='', created_at=0.0, error=None, incomplete_details=None, instructions=None, metadata=None, model='', object='response', output=[], parallel_tool_calls=False, temperature=None, tool_choice='none', tools=[], top_p=None, background=None, completed_at=None, conversation=None, max_output_tokens=None, max_tool_calls=None, moderation=None, previous_response_id=None, prompt=None, prompt_cache_key=None, prompt_cache_options=None, prompt_cache_retention=None, reasoning=None, safety_identifier=None, service_tier=None, status=None, text=None, top_logprobs=None, truncation=None, usage=None, user=None), sequence_number=0, type='response.created')
ResponseOutputItemAddedEvent(item=ResponseReasoningItem(id='', summary=[], type='reasoning', content=None, encrypted_content=None, status='in_progress'), output_index=0, sequence_number=1, type='response.output_item.added')
ResponseReasoningSummaryTextDeltaEvent(delta='I', item_id='', output_index=0, sequence_number